# Figure 3 — Pan-Cancer Comparison of Pathway Coordination

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aistanbulresearch/msep/blob/main/notebooks/figures/figure3_pan_cancer.ipynb)

> **Status:** skeleton only. The full CellxGene Census pull for 20 cancer types lives in the WP-3 pan-cancer notebook ([`notebooks/pan_cancer/fetch_and_profile.ipynb`](../pan_cancer/fetch_and_profile.ipynb), in progress). Once that notebook exports the CV matrix, this notebook consumes it to render Figure 3's three panels.

Reproduces **Figure 3** from Çavuş & Kuşkucu (2026): a 12-cancer-type (20-type post-WP-3) comparison placing chordoma as the most EMT-coordinated malignancy.

**Panels**

| Panel | What it shows | Source |
|---|---|---|
| A | Heatmap of across-cells CV for all cancer types × three resistance pathways | CellxGene Census queries (WP-3) |
| B | EMT CV horizontal bar ranking, chordoma highlighted | `msep.plot_pan_cancer` |
| C | Chordoma pathway CV vs pan-cancer median | derived from Panel A |

## 1. Install

In [ ]:
!pip install -q --upgrade-strategy only-if-needed 'msep[plotting]>=1.1.0'

## 2. Demo — pan-cancer CV from published values

This skeleton uses the 12 CV values reported in the paper's Table S5 directly, so the ranking plot is immediately reproducible. When the WP-3 Census pipeline lands, replace this hard-coded DataFrame with the live CV matrix.

In [ ]:
import pandas as pd
import msep

# From Çavuş & Kuşkucu (2026) Figure 3B / Supplementary Table S5
pan_cancer = pd.DataFrame({
    'cancer_type': [
        'chordoma', 'osteosarcoma', 'basal_cell_carcinoma', 'breast_cancer',
        'glioblastoma', 'triple_negative_breast', 'invasive_ductal_breast',
        'nonpapillary_renal', 'squamous_lung', 'uveal_melanoma',
        'lung_adenocarcinoma', 'melanoma_smartseq2',
    ],
    'cv_emt': [4.632, 5.25, 5.49, 5.49, 6.72, 7.64, 7.83, 6.95, 6.12, 6.34, 6.88, 4.85],
    'cv_ferroptosis': [5.387, 5.80, 6.21, 6.45, 6.10, 7.88, 7.34, 7.22, 6.98, 5.92, 6.64, 5.65],
    'cv_immune_evasion': [8.548, 8.12, 8.02, 8.31, 9.52, 9.33, 8.74, 8.57, 8.90, 7.99, 8.45, 7.85],
})
pan_cancer

## 3. Panel A — CV heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

mat = pan_cancer.set_index('cancer_type')[['cv_emt', 'cv_ferroptosis', 'cv_immune_evasion']]
mat.columns = ['EMT', 'Ferroptosis', 'Immune evasion']
mat = mat.sort_values('EMT')

fig_a, ax_a = plt.subplots(figsize=(7, 6))
sns.heatmap(mat, annot=True, fmt='.2f', cmap='YlOrRd_r', linewidths=0.5,
            vmin=4, vmax=10, cbar_kws={'label': 'CV'}, ax=ax_a)
ax_a.set_title('A · Pan-cancer pathway CV (lower = more coordinated)')
ax_a.set_ylabel('')
fig_a.tight_layout()
fig_a.savefig('figure3_panelA_heatmap.pdf', bbox_inches='tight')
fig_a.savefig('figure3_panelA_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Panel B — EMT CV ranking with chordoma highlighted

In [ ]:
emt_df = pan_cancer[['cancer_type', 'cv_emt']].rename(columns={'cv_emt': 'cv'})
fig_b = msep.plot_pan_cancer(emt_df, pathway='emt', highlight='chordoma', figsize=(8, 5))
fig_b.savefig('figure3_panelB_emt_ranking.pdf', bbox_inches='tight')
fig_b.savefig('figure3_panelB_emt_ranking.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Panel C — Chordoma vs pan-cancer median

In [ ]:
medians = pan_cancer.drop(columns='cancer_type').median()
chordoma = pan_cancer.query('cancer_type == "chordoma"').iloc[0].drop('cancer_type')
cmp = pd.DataFrame({'pan_cancer_median': medians, 'chordoma': chordoma})
cmp.index = ['EMT', 'Ferroptosis', 'Immune evasion']

fig_c, ax_c = plt.subplots(figsize=(6, 4))
x = range(len(cmp))
w = 0.35
ax_c.bar([i - w/2 for i in x], cmp['pan_cancer_median'], w, label='Pan-cancer median', color='#7F8C8D')
ax_c.bar([i + w/2 for i in x], cmp['chordoma'], w, label='Chordoma', color='#E74C3C')
ax_c.set_xticks(list(x))
ax_c.set_xticklabels(cmp.index)
ax_c.set_ylabel('Across-cells CV')
ax_c.set_title('C · Chordoma vs pan-cancer median')
ax_c.legend()
ax_c.spines[['top', 'right']].set_visible(False)
fig_c.tight_layout()
fig_c.savefig('figure3_panelC_vs_median.pdf', bbox_inches='tight')
fig_c.savefig('figure3_panelC_vs_median.png', dpi=300, bbox_inches='tight')
plt.show()

## TODO (WP-3 integration)

- Replace the hard-coded `pan_cancer` DataFrame in Section 2 with output from `notebooks/pan_cancer/fetch_and_profile.ipynb` (8 new cancer types → 20-type panel).
- Re-run all three panels — the plotting code is dataset-agnostic.
- Update the chordoma highlight if the pan-cancer ranking shifts.